# 2026 Final Four Analytics Challenge

## Overview
Since its inception in 1939, the NCAA basketball tournament, commonly known as the Final Four, has grown into the pinnacle of collegiate athletics. Over the past 86 years, the tournament has evolved from a modest field of 8 teams to the now iconic 64-team format, with a matching women's tournament added in 1982.

The 2026 NCAA tournament is a single game elimination style tournament that begins March 17 and culminates with the national championship game on April 6. Its fast-paced intensity and unpredictable outcomes ignite school spirit and captivate sports fans across the world, earning the fitting nickname March Madness.

March Madness is more than thrilling athletic competition. It is a nationwide tradition of prediction and participation. From forecasting which teams will make the tournament and how they will be seeded, to anticipating upsets and choosing a champion, bracket-building has become an interactive way for fans to feel part of the action. Each spring, nearly 100 million people fill out March Madness brackets, hoping their favorite teams will survive and advance.

The unpredictable nature of athletic competition makes forecasting the Final Four tournament exceptionally difficult. In fact, the NCAA estimates the odds of selecting every tournament team and crafting a perfect bracket are 1 in 9.2 quintillion. As a result, the NCAA is increasingly interested in exploring the dynamics that influence tournament success.

## Problem Statement
Selection Sunday is the moment the entire college basketball world holds its breath, when dreams are made, bubbles burst, and the road to March Madness officially begins. It is the dramatic, high-energy kickoff to the NCAA Tournament.

The first round of the 2026 Final Four Analytics challenge focuses on the dynamic process involved with selection and seeding in the tournament. Students are challenged to use machine learning to predict whether a team will make the tournament and its corresponding seed.

## Notebook Goal
This notebook builds a machine learning pipeline to:
1. Clean and engineer team-level features.
2. Predict tournament selection likelihood.
3. Rank teams for final seed submission output.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [3]:
df = pd.read_csv('NCAA_Seed_Training_Set2.0.csv')
df.head()

,RecordID,Season,Team,Conference,Overall Seed,Bid Type,NET Rank,PrevNET,AvgOppNETRank,AvgOppNET,WL,Conf.Record,Non-ConferenceRecord,RoadWL,NETSOS,NETNonConfSOS,Quadrant1,Quadrant2,Quadrant3,Quadrant4
0,2020-21-UCDavis,2020-21,UC Davis,Big West,NaN,NaN,223.0,224.0,240.0,211.0,8-Sep,5-Jul,3-Feb,3-Apr,264.0,287.0,0-0,0-3,2-Jan,3-Aug
1,2020-21-MichiganSt.,2020-21,Michigan St.,Big Ten,43.0,AL,70.0,70.0,20.0,75.0,15-12,12-Sep,Jun-00,8-Mar,9.0,238.0,10-May,2-Apr,Feb-00,Apr-00
2,2020-21-ULM,2020-21,ULM,Sun Belt,NaN,NaN,292.0,292.0,244.0,214.0,19-Jul,14-May,5-Feb,10-Feb,313.0,263.0,0-0,0-1,7-Feb,11-May
3,2020-21-CentralConn.St.,2020-21,Central Conn. St.,NEC,NaN,NaN,301.0,301.0,200.0,195.0,16-May,13-May,0-3,9-Feb,242.0,54.0,0-1,0-2,6-Feb,7-Mar
4,2020-21-Colgate,2020-21,Colgate,Patriot,57.0,AQ,9.0,9.0,154.0,169.0,14-1,14-1,0-0,Jun-00,257.0,NaN,0-0,Feb-00,1-Jun,Jun-00


In [5]:
import pandas as pd
import re
import calendar

# Read as strings so nothing else gets auto-converted
df = pd.read_csv("NCAA_Seed_Training_Set2.0.csv", dtype=str)

quad_cols = ["WL","Conf.Record","Non-ConferenceRecord","RoadWL","Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]
month_map = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def fix_excel_date_like_score(x):
    if pd.isna(x):
        return x
    s = str(x).strip()

    # Already correct score format, e.g. 3-2
    if re.fullmatch(r"\d{1,2}-\d{1,2}", s):
        return s

    # Pattern like 2-Jan -> 1-2 (month-day)
    m = re.fullmatch(r"(\d{1,2})-([A-Za-z]{3})", s)
    if m:
        day = int(m.group(1))
        mon = month_map.get(m.group(2).lower())
        if mon is not None:
            return f"{mon}-{day}"

    # Pattern like Apr-00 -> 4-0 (month-second number)
    m = re.fullmatch(r"([A-Za-z]{3})-(\d{2})", s)
    if m:
        mon = month_map.get(m.group(1).lower())
        second = int(m.group(2))
        if mon is not None:
            return f"{mon}-{second}"

    return s

for c in quad_cols:
    df[c] = df[c].apply(fix_excel_date_like_score)

# Optional: quick check for anything still date-like
for c in quad_cols:
    bad = df[df[c].str.match(r"^\d{1,2}-[A-Za-z]{3}$|^[A-Za-z]{3}-\d{2}$", na=False)][c].unique()
    if len(bad):
        print(c, bad[:10])

# df.to_csv("NCAA_Seed_Training_Set2.0_fixed.csv", index=False)
df.head()

,RecordID,Season,Team,Conference,Overall Seed,Bid Type,NET Rank,PrevNET,AvgOppNETRank,AvgOppNET,WL,Conf.Record,Non-ConferenceRecord,RoadWL,NETSOS,NETNonConfSOS,Quadrant1,Quadrant2,Quadrant3,Quadrant4
0,2020-21-UCDavis,2020-21,UC Davis,Big West,NaN,NaN,223,224,240,211,9-8,7-5,2-3,4-3,264,287,0-0,0-3,1-2,8-3
1,2020-21-MichiganSt.,2020-21,Michigan St.,Big Ten,43,AL,70,70,20,75,15-12,9-12,6-0,3-8,9,238,5-10,4-2,2-0,4-0
2,2020-21-ULM,2020-21,ULM,Sun Belt,NaN,NaN,292,292,244,214,7-19,5-14,2-5,2-10,313,263,0-0,0-1,2-7,5-11
3,2020-21-CentralConn.St.,2020-21,Central Conn. St.,NEC,NaN,NaN,301,301,200,195,5-16,5-13,0-3,2-9,242,54,0-1,0-2,2-6,3-7
4,2020-21-Colgate,2020-21,Colgate,Patriot,57,AQ,9,9,154,169,14-1,14-1,0-0,6-0,257,NaN,0-0,2-0,6-1,6-0


In [15]:
df['Bid Type'].value_counts()

Bid Type
AL    140
AQ    109
Name: count, dtype: int64

In [6]:
import pandas as pd
import re
import calendar

# Read as strings so nothing else gets auto-converted
df_test = pd.read_csv("NCAA_Seed_Test_Set2.0.csv", dtype=str)

quad_cols = ["WL","Conf.Record","Non-ConferenceRecord","RoadWL","Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]
month_map = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def fix_excel_date_like_score(x):
    if pd.isna(x):
        return x
    s = str(x).strip()

    # Already correct score format, e.g. 3-2
    if re.fullmatch(r"\d{1,2}-\d{1,2}", s):
        return s

    # Pattern like 2-Jan -> 1-2 (month-day)
    m = re.fullmatch(r"(\d{1,2})-([A-Za-z]{3})", s)
    if m:
        day = int(m.group(1))
        mon = month_map.get(m.group(2).lower())
        if mon is not None:
            return f"{mon}-{day}"

    # Pattern like Apr-00 -> 4-0 (month-second number)
    m = re.fullmatch(r"([A-Za-z]{3})-(\d{2})", s)
    if m:
        mon = month_map.get(m.group(1).lower())
        second = int(m.group(2))
        if mon is not None:
            return f"{mon}-{second}"

    return s

for c in quad_cols:
    df_test[c] = df_test[c].apply(fix_excel_date_like_score)

# Optional: quick check for anything still date-like
for c in quad_cols:
    bad = df_test[df_test[c].str.match(r"^\d{1,2}-[A-Za-z]{3}$|^[A-Za-z]{3}-\d{2}$", na=False)][c].unique()
    if len(bad):
        print(c, bad[:10])

# df_test.to_csv("NCAA_Seed_Test_Set2.0_fixed.csv", index=False)
df_test.head()

,RecordID,Season,Team,Conference,Bid Type,NET Rank,PrevNET,AvgOppNETRank,AvgOppNET,WL,Conf.Record,Non-ConferenceRecord,RoadWL,NETSOS,NETNonConfSOS,Quadrant1,Quadrant2,Quadrant3,Quadrant4
0,2020-21-Baylor,2020-21,Baylor,Big 12,AL,2,2,86,124,22-2,14-2,8-0,7-1,50,211,8-2,2-0,7-0,5-0
1,2020-21-Arkansas,2020-21,Arkansas,SEC,AL,14,14,61,102,22-6,14-5,8-1,5-4,47,227,6-5,6-1,5-0,5-0
2,2020-21-Purdue,2020-21,Purdue,Big Ten,AL,22,22,11,69,18-9,13-7,5-2,5-6,17,176,6-7,7-1,3-1,2-0
3,2020-21-OklahomaSt.,2020-21,Oklahoma St.,Big 12,AL,29,29,30,86,20-8,13-8,7-0,8-4,14,155,10-6,2-0,3-2,5-0
4,2020-21-SouthernCalifornia,2020-21,Southern California,Pac-12,AL,19,19,47,94,22-7,16-6,6-1,7-3,75,136,3-4,6-3,10-0,3-0


In [7]:
import pandas as pd
import numpy as np
import re
import calendar

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [8]:
# 2) Feature engineering from records like "18-7"
def split_wl(col, prefix):
    wins = []
    losses = []
    for v in col.fillna("0-0"):
        m = re.fullmatch(r"\s*(\d+)\s*-\s*(\d+)\s*", str(v))
        if m:
            w, l = int(m.group(1)), int(m.group(2))
        else:
            w, l = np.nan, np.nan
        wins.append(w)
        losses.append(l)
    return pd.Series(wins), pd.Series(losses)

def add_record_features(df_):
    out = df_.copy()

    # overall WL
    w, l = split_wl(out["WL"], "overall")
    out["overall_wins"] = w
    out["overall_losses"] = l
    out["overall_win_pct"] = w / (w + l)

    # conference
    w, l = split_wl(out["Conf.Record"], "conf")
    out["conf_wins"] = w
    out["conf_losses"] = l
    out["conf_win_pct"] = w / (w + l)

    # non-conference
    w, l = split_wl(out["Non-ConferenceRecord"], "nc")
    out["nc_wins"] = w
    out["nc_losses"] = l
    out["nc_win_pct"] = w / (w + l)

    # road
    w, l = split_wl(out["RoadWL"], "road")
    out["road_wins"] = w
    out["road_losses"] = l
    out["road_win_pct"] = w / (w + l)

    # quadrant features
    for q in ["Quadrant1","Quadrant2","Quadrant3","Quadrant4"]:
        w, l = split_wl(out[q], q.lower())
        out[f"{q}_wins"] = w
        out[f"{q}_losses"] = l

    return out

df2 = add_record_features(df)

In [9]:
# 3) Targets
# MadeTournament: Bid Type not null/blank means selected (AQ/AL)
df2["MadeTournament"] = (~df2["Bid Type"].isna()) & (df2["Bid Type"].str.strip() != "")
df2["MadeTournament"] = df2["MadeTournament"].astype(int)

# Seed target only for selected teams
df2["SeedNum"] = pd.to_numeric(df2["Overall Seed"], errors="coerce")

In [10]:
# 4) Select features
candidate_features = [
    "Conference",
    "NET Rank", "PrevNET", "AvgOppNET Rank", "AvgOppNET", "NETSOS", "NETNonConfSOS",
    "overall_wins","overall_losses","overall_win_pct",
    "conf_wins","conf_losses","conf_win_pct",
    "nc_wins","nc_losses","nc_win_pct",
    "road_wins","road_losses","road_win_pct",
    "Quadrant1_wins","Quadrant1_losses",
    "Quadrant2_wins","Quadrant2_losses",
    "Quadrant3_wins","Quadrant3_losses",
    "Quadrant4_wins","Quadrant4_losses"
]
candidate_features = [c for c in candidate_features if c in df2.columns]

X = df2[candidate_features]
y_cls = df2["MadeTournament"]

num_cols = [c for c in candidate_features if c != "Conference"]
cat_cols = [c for c in candidate_features if c == "Conference"]

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop"
)

In [29]:
# 5) Classification model: who gets in?
from sklearn.base import clone

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

clf = Pipeline([
    ("pre", clone(pre)),
    ("model", RandomForestClassifier(
        n_estimators=400, random_state=42, class_weight="balanced"
    ))
])

clf.fit(X_train, y_train)
pred_cls = clf.predict(X_test)
proba_cls = clf.predict_proba(X_test)[:,1]

print("Classification Report")
print(classification_report(y_test, pred_cls))
print("Confusion Matrix")
print(confusion_matrix(y_test, pred_cls))
print("ROC-AUC:", roc_auc_score(y_test, proba_cls))

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       221
           1       0.88      0.74      0.80        50

    accuracy                           0.93       271
   macro avg       0.91      0.86      0.88       271
weighted avg       0.93      0.93      0.93       271

Confusion Matrix
[[216   5]
 [ 13  37]]
ROC-AUC: 0.9828506787330317


In [48]:
# 6) Regression model: seed number for teams that made tournament
from sklearn.base import clone

seed_df = df2[df2["SeedNum"].notna()].copy()
X_seed = seed_df[candidate_features]
y_seed = seed_df["SeedNum"]

Xtr, Xte, ytr, yte = train_test_split(X_seed, y_seed, test_size=0.2, random_state=42)

reg = Pipeline([
    ("pre", clone(pre)),
    ("model", RandomForestRegressor(n_estimators=500, random_state=42))
])

reg.fit(Xtr, ytr)
pred_seed = reg.predict(Xte)

seed_mae = mean_absolute_error(yte, pred_seed)
seed_rmse = np.sqrt(mean_squared_error(yte, pred_seed))
seed_r2 = r2_score(yte, pred_seed)

print("Seed MAE:", round(seed_mae, 4))
print("Seed RMSE:", round(seed_rmse, 4))
print("Seed R2:", round(seed_r2, 4))

Seed MAE: 3.8403
Seed RMSE: 5.9854
Seed R2: 0.9135


In [49]:
# 6a) Seed model improvement: compare regressors and keep the best by RMSE
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor

seed_candidates = {
    "RF_500": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),
    "RF_1200_leaf2": RandomForestRegressor(
        n_estimators=1200,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees_1200": ExtraTreesRegressor(
        n_estimators=1200,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "GBR": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ),
}

seed_results = []
best_name = None
best_rmse = float("inf")
best_pipe = None

for name, model in seed_candidates.items():
    pipe = Pipeline([
        ("pre", clone(pre)),
        ("model", model)
    ])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)

    mae = mean_absolute_error(yte, pred)
    rmse = np.sqrt(mean_squared_error(yte, pred))
    r2 = r2_score(yte, pred)
    seed_results.append((name, mae, rmse, r2))

    if rmse < best_rmse:
        best_rmse = rmse
        best_name = name
        best_pipe = pipe

seed_results_df = pd.DataFrame(seed_results, columns=["model", "MAE", "RMSE", "R2"]).sort_values("RMSE")
print(seed_results_df.to_string(index=False))
print("\nBest seed model by RMSE:", best_name, "| RMSE:", round(best_rmse, 4))

# Use best model for downstream ranking/submission cells
reg = best_pipe

          model      MAE     RMSE       R2
         RF_500 3.840280 5.985416 0.913496
ExtraTrees_1200 4.326678 5.986646 0.913461
  RF_1200_leaf2 4.036177 6.098267 0.910203
            GBR 4.147253 6.286782 0.904566

Best seed model by RMSE: RF_500 | RMSE: 5.9854


In [45]:
# 6b) Holdout ranking evaluation (before final submission prediction)
from scipy.stats import spearmanr

print("Is PrevNET in feature list?", "PrevNET" in candidate_features)

# Build holdout frame from the same split used in classification evaluation
holdout = df2.loc[X_test.index, ["RecordID", "Season", "SeedNum"]].copy()
holdout["SelProb"] = clf.predict_proba(X_test)[:, 1]
holdout["SeedPred"] = reg.predict(X_test)

# Use tuned weight if available, otherwise default to 0.55
w_eval = float(best_w) if "best_w" in globals() else 0.55

# Rank all holdout teams within each season (higher score -> better rank)
holdout_parts = []
for season, g in holdout.groupby("Season", sort=True):
    g = g.copy()

    p = g["SelProb"]
    s = g["SeedPred"]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    score = w_eval * p_norm + (1.0 - w_eval) * (1.0 - s_norm)

    g["PredRank"] = score.rank(method="first", ascending=False).astype(int)
    holdout_parts.append(g)

holdout_ranked = pd.concat(holdout_parts, ignore_index=True)

# Evaluate ranking only on teams that truly made tournament (have true seed)
eval_rank = holdout_ranked[holdout_ranked["SeedNum"].notna()].copy()

if len(eval_rank) > 1:
    overall_spearman = spearmanr(eval_rank["PredRank"], eval_rank["SeedNum"], nan_policy="omit").correlation
    print("Holdout ranking Spearman (overall):", round(float(overall_spearman), 4))

    season_corrs = []
    for season, d in eval_rank.groupby("Season"):
        if len(d) > 1:
            corr = spearmanr(d["PredRank"], d["SeedNum"], nan_policy="omit").correlation
            if not np.isnan(corr):
                season_corrs.append(float(corr))

    if season_corrs:
        print("Holdout ranking Spearman (season avg):", round(float(np.mean(season_corrs)), 4))
    else:
        print("Holdout ranking Spearman (season avg): not enough per-season seeded teams")

    print("Teams with true seeds in holdout:", len(eval_rank))
else:
    print("Not enough seeded teams in holdout to compute ranking correlation.")

eval_rank[["RecordID", "Season", "SeedNum", "PredRank", "SelProb", "SeedPred"]].head(20)

Is PrevNET in feature list? True
Holdout ranking Spearman (overall): 0.8854
Holdout ranking Spearman (season avg): 0.922
Teams with true seeds in holdout: 50


,RecordID,Season,SeedNum,PredRank,SelProb,SeedPred
2,2020-21-Colorado,2020-21,20.0,3,0.9075,19.144
7,2020-21-Texas,2020-21,11.0,2,0.9875,12.454
11,2020-21-Iona,2020-21,62.0,22,0.1075,60.592
12,2020-21-MichiganSt.,2020-21,43.0,10,0.1575,41.962
19,2020-21-VirginiaTech,2020-21,38.0,6,0.3875,41.294
24,2020-21-MoreheadSt.,2020-21,56.0,7,0.5575,57.466
25,2020-21-Gonzaga,2020-21,1.0,1,0.9775,6.360
27,2020-21-TexasSouthern,2020-21,66.0,15,0.2950,65.626
29,2020-21-Maryland,2020-21,36.0,5,0.5775,36.750
33,2020-21-WichitaSt.,2020-21,45.0,9,0.2375,43.930


In [31]:
# 7) Final submission output on test set: RecordID, Overall Seed
# Build test features using the same feature engineering as training.
df_test2 = add_record_features(df_test)
X_submit = df_test2[candidate_features]

# Stage 1: predict tournament selection probability
submit_sel_prob = clf.predict_proba(X_submit)[:, 1]
submit_sel_pred = (submit_sel_prob >= 0.5).astype(int)

# Stage 2: predict seed (only keep seed when selected)
submit_seed_pred = reg.predict(X_submit)
submit_seed = np.where(submit_sel_pred == 1, np.clip(np.round(submit_seed_pred), 1, 16), np.nan)

submission = pd.DataFrame({
    "RecordID": df_test2["RecordID"],
    "Overall Seed": submit_seed
})

# If your challenge requires integer seeds only, keep only predicted tournament teams.
submission = submission.dropna(subset=["Overall Seed"]).copy()
submission["Overall Seed"] = submission["Overall Seed"].astype(int)

submission.head(20)

,RecordID,Overall Seed
0,2020-21-Baylor,6
1,2020-21-Arkansas,16
2,2020-21-Purdue,16
3,2020-21-OklahomaSt.,16
4,2020-21-SouthernCalifornia,16
5,2020-21-TexasTech,16
6,2020-21-Wisconsin,16
7,2020-21-Syracuse,16
8,2020-21-UCLA,16
9,2020-21-Winthrop,16


In [27]:
from sklearn.base import clone

# keep this as your base transformer definition
pre_base = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop"
)

clf = Pipeline([
    ("pre", clone(pre_base)),
    ("model", RandomForestClassifier(
        n_estimators=400, random_state=42, class_weight="balanced"
    ))
])

reg = Pipeline([
    ("pre", clone(pre_base)),
    ("model", RandomForestRegressor(n_estimators=500, random_state=42))
])

In [38]:
# Build base scoring frame
submission_raw = pd.DataFrame({
    "RecordID": df_test2["RecordID"],
    "SelProb": submit_sel_prob,
    "SeedPred": submit_seed_pred
})

# Season from RecordID like "2020-21-Baylor"
submission_raw["Season"] = submission_raw["RecordID"].str.extract(r"^(\d{4}-\d{2})")

rows = []
for season, g in submission_raw.groupby("Season", sort=True):
    # Higher SelProb is better; lower SeedPred is better
    g = g.sort_values(["SelProb", "SeedPred"], ascending=[False, True]).copy()

    # Rank all teams in that season to keep full row coverage.
    g["Overall Seed"] = np.arange(1, len(g) + 1)

    rows.append(g[["RecordID", "Season", "Overall Seed"]])

submission = pd.concat(rows, ignore_index=True)

# Final output schema required by challenge
submission = submission[["RecordID", "Overall Seed"]]
submission.head(20)

,RecordID,Overall Seed
0,2020-21-Arkansas,1
1,2020-21-SouthernCalifornia,2
2,2020-21-Baylor,3
3,2020-21-Purdue,4
4,2020-21-OklahomaSt.,5
5,2020-21-Wisconsin,6
6,2020-21-TexasTech,7
7,2020-21-Syracuse,8
8,2020-21-Winthrop,9
9,2020-21-UCLA,10


In [39]:
check = submission.assign(
    Season=submission["RecordID"].str.extract(r"^(\d{4}-\d{2})")
)

# 1) No duplicate rank in same season
dupes = check.duplicated(["Season", "Overall Seed"]).sum()
print("Duplicate ranks within season:", dupes)

# 2) Coverage per season
coverage = check.groupby("Season")["Overall Seed"].agg(["count", "min", "max", "nunique"])
print(coverage)

Duplicate ranks within season: 0
         count  min  max  nunique
Season                           
2020-21     89    1   89       89
2021-22     90    1   90       90
2022-23     91    1   91       91
2023-24     90    1   90       90
2024-25     91    1   91       91


In [40]:
submission.to_csv("NCAA_Seed_Submission.csv", index=False)

In [41]:
submission.shape

(451, 2)

In [42]:
submission

,RecordID,Overall Seed
0,2020-21-Arkansas,1
1,2020-21-SouthernCalifornia,2
2,2020-21-Baylor,3
3,2020-21-Purdue,4
4,2020-21-OklahomaSt.,5
...,...,...
446,2024-25-HolyCross,87
447,2024-25-UMBC,88
448,2024-25-SouthernInd.,89
449,2024-25-NewHampshire,90


In [43]:
# Improved end-to-end pipeline: season-aware tuning + stronger ranking submission
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# Rebuild engineered datasets
train_df = add_record_features(df.copy())
test_df = add_record_features(df_test.copy())

# Targets
train_df["MadeTournament"] = (~train_df["Bid Type"].isna()) & (train_df["Bid Type"].str.strip() != "")
train_df["MadeTournament"] = train_df["MadeTournament"].astype(int)
train_df["SeedNum"] = pd.to_numeric(train_df["Overall Seed"], errors="coerce")

# Feature set
features = [
    "Conference",
    "NET Rank", "PrevNET", "AvgOppNET Rank", "AvgOppNET", "NETSOS", "NETNonConfSOS",
    "overall_wins", "overall_losses", "overall_win_pct",
    "conf_wins", "conf_losses", "conf_win_pct",
    "nc_wins", "nc_losses", "nc_win_pct",
    "road_wins", "road_losses", "road_win_pct",
    "Quadrant1_wins", "Quadrant1_losses",
    "Quadrant2_wins", "Quadrant2_losses",
    "Quadrant3_wins", "Quadrant3_losses",
    "Quadrant4_wins", "Quadrant4_losses"
]
features = [c for c in features if c in train_df.columns]

X_all = train_df[features]
X_submit_all = test_df[features]
y_sel = train_df["MadeTournament"]
seed_mask = train_df["SeedNum"].notna()
y_seed = train_df.loc[seed_mask, "SeedNum"]


def make_preprocessor():
    num_cols = [c for c in features if c != "Conference"]
    cat_cols = [c for c in features if c == "Conference"]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ],
        remainder="drop",
    )


def fit_models(train_idx):
    clf_local = Pipeline([
        ("pre", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=700,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    reg_local = Pipeline([
        ("pre", make_preprocessor()),
        ("model", ExtraTreesRegressor(
            n_estimators=900,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    clf_local.fit(X_all.iloc[train_idx], y_sel.iloc[train_idx])

    seed_train_idx = np.intersect1d(train_idx, np.where(seed_mask.values)[0])
    reg_local.fit(X_all.iloc[seed_train_idx], train_df.iloc[seed_train_idx]["SeedNum"])
    return clf_local, reg_local


def make_rank(df_part, w):
    # Normalize components within season to make blend robust season-to-season.
    p = df_part["SelProb"]
    s = df_part["SeedPred"]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)

    score = w * p_norm + (1.0 - w) * (1.0 - s_norm)
    return score.rank(method="first", ascending=False).astype(int)


# Season-aware tuning for blend weight
season_values = sorted(train_df["Season"].dropna().unique())
weight_grid = np.linspace(0.55, 0.95, 9)
cv_scores = {float(w): [] for w in weight_grid}

for hold in season_values:
    tr_idx = np.where(train_df["Season"].values != hold)[0]
    va_idx = np.where(train_df["Season"].values == hold)[0]

    if len(tr_idx) == 0 or len(va_idx) == 0:
        continue

    clf_cv, reg_cv = fit_models(tr_idx)

    fold = train_df.iloc[va_idx][["RecordID", "Season", "SeedNum"]].copy()
    fold["SelProb"] = clf_cv.predict_proba(X_all.iloc[va_idx])[:, 1]
    fold["SeedPred"] = reg_cv.predict(X_all.iloc[va_idx])

    eval_fold = fold[fold["SeedNum"].notna()].copy()
    if eval_fold.empty:
        continue

    for w in weight_grid:
        fold_rank = make_rank(fold, float(w))
        eval_fold["PredRank"] = fold_rank.loc[eval_fold.index]
        corr = spearmanr(eval_fold["PredRank"], eval_fold["SeedNum"], nan_policy="omit").correlation
        if np.isnan(corr):
            corr = -1.0
        cv_scores[float(w)].append(float(corr))

avg_scores = {w: (np.mean(v) if len(v) else -1.0) for w, v in cv_scores.items()}
best_w = max(avg_scores, key=avg_scores.get)

print("Best blend weight (selection probability):", round(best_w, 3))
print("CV Spearman by weight:", {round(k, 2): round(v, 4) for k, v in avg_scores.items()})

# Fit final models on all training data
all_idx = np.arange(len(train_df))
clf_final, reg_final = fit_models(all_idx)

# Predict on full test set
submit = test_df[["RecordID", "Season"]].copy()
submit["SelProb"] = clf_final.predict_proba(X_submit_all)[:, 1]
submit["SeedPred"] = reg_final.predict(X_submit_all)

# Per-season unique ranking for all rows
ranked_parts = []
for season, part in submit.groupby("Season", sort=True):
    part = part.copy()
    part["Overall Seed"] = make_rank(part, best_w)
    ranked_parts.append(part[["RecordID", "Overall Seed"]])

submission = pd.concat(ranked_parts, ignore_index=True)
submission["Overall Seed"] = submission["Overall Seed"].astype(int)

# Validation checks
assert submission.shape[0] == len(test_df), f"Expected {len(test_df)} rows, got {submission.shape[0]}"
assert submission["RecordID"].nunique() == len(test_df), "Duplicate RecordID found in submission"

check = submission.assign(Season=submission["RecordID"].str.extract(r"^(\d{4}-\d{2})"))
dupes = check.duplicated(["Season", "Overall Seed"]).sum()
print("Duplicate ranks within season:", int(dupes))
print("Final submission shape:", submission.shape)

submission.to_csv("NCAA_Seed_Submission_improved.csv", index=False)
submission.head(20)

Best blend weight (selection probability): 0.55
CV Spearman by weight: {0.55: np.float64(0.9509), 0.6: np.float64(0.9469), 0.65: np.float64(0.9417), 0.7: np.float64(0.9371), 0.75: np.float64(0.9318), 0.8: np.float64(0.9242), 0.85: np.float64(0.917), 0.9: np.float64(0.9064), 0.95: np.float64(0.8945)}
Duplicate ranks within season: 0
Final submission shape: (451, 2)


,RecordID,Overall Seed
0,2020-21-Baylor,1
1,2020-21-Arkansas,2
2,2020-21-Purdue,3
3,2020-21-OklahomaSt.,4
4,2020-21-SouthernCalifornia,5
5,2020-21-TexasTech,7
6,2020-21-Wisconsin,6
7,2020-21-Syracuse,9
8,2020-21-UCLA,10
9,2020-21-Winthrop,8


In [53]:
# Final tuned submission: RandomizedSearchCV for seed model + top-68 ranking per season
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Ensure engineered test set exists
if "df_test2" not in globals():
    df_test2 = add_record_features(df_test)

# Train classifier on full training data for final selection signal
clf_full = Pipeline([
    ("pre", clone(pre)),
    ("model", RandomForestClassifier(
        n_estimators=800,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ))
])
clf_full.fit(X, y_cls)

# Tune seed regressor with randomized search
seed_search_pipe = Pipeline([
    ("pre", clone(pre)),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_distributions = {
    "model__n_estimators": randint(400, 1800),
    "model__max_depth": [None, 6, 8, 10, 12, 16, 20],
    "model__min_samples_split": randint(2, 16),
    "model__min_samples_leaf": randint(1, 8),
    "model__max_features": ["sqrt", "log2", 0.5, 0.7, 1.0],
}

seed_search = RandomizedSearchCV(
    estimator=seed_search_pipe,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)
seed_search.fit(X_seed, y_seed)

reg_tuned = seed_search.best_estimator_
print("Best seed CV RMSE:", round(-seed_search.best_score_, 4))
print("Best seed params:", seed_search.best_params_)

# Build test predictions
X_submit_tuned = df_test2[candidate_features]
submit_tmp = df_test2[["RecordID"]].copy()
submit_tmp["Season"] = submit_tmp["RecordID"].str.extract(r"^(\d{4}-\d{2})")
submit_tmp["SelProb"] = clf_full.predict_proba(X_submit_tuned)[:, 1]
submit_tmp["SeedPred"] = reg_tuned.predict(X_submit_tuned)

# Use tuned blending weight if available from season-aware analysis
blend_w = float(best_w) if "best_w" in globals() else 0.55
TOP_N = 68

final_parts = []
for season, part in submit_tmp.groupby("Season", sort=True):
    part = part.copy()

    p = part["SelProb"]
    s = part["SeedPred"]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)

    score = blend_w * p_norm + (1.0 - blend_w) * (1.0 - s_norm)
    part["_rank_all"] = score.rank(method="first", ascending=False).astype(int)

    # Keep only top 68 as ranks 1..68; rest are null.
    part = part.sort_values("_rank_all").copy()
    part["Overall Seed"] = np.nan
    keep_n = min(TOP_N, len(part))
    part.iloc[:keep_n, part.columns.get_loc("Overall Seed")] = np.arange(1, keep_n + 1)

    final_parts.append(part[["RecordID", "Season", "Overall Seed"]])

submission_tuned_full = pd.concat(final_parts, ignore_index=True)
submission_tuned_full = submission_tuned_full.sort_values("RecordID").reset_index(drop=True)

# Required validation checks
assert submission_tuned_full.shape[0] == 451, f"Expected 451 rows, got {submission_tuned_full.shape[0]}"
assert submission_tuned_full["RecordID"].nunique() == 451, "RecordID duplicates found"

non_null = submission_tuned_full[submission_tuned_full["Overall Seed"].notna()].copy()
dupes_tuned = non_null.duplicated(["Season", "Overall Seed"]).sum()
coverage_tuned = submission_tuned_full.groupby("Season")["Overall Seed"].apply(lambda x: x.notna().sum())

print("Duplicate non-null ranks within season:", int(dupes_tuned))
print("Non-null ranks per season:")
print(coverage_tuned)

# Final file in required schema
submission_tuned = submission_tuned_full[["RecordID", "Overall Seed"]].copy()
print("Final tuned submission shape:", submission_tuned.shape)

submission_tuned.to_csv("NCAA_Seed_Submission_tuned_top68.csv", index=False)
submission_tuned.head(20)

Best seed CV RMSE: 5.1541
Best seed params: {'model__max_depth': 8, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 1012}
Duplicate non-null ranks within season: 0
Non-null ranks per season:
Season
2020-21    68
2021-22    68
2022-23    68
2023-24    68
2024-25    68
Name: Overall Seed, dtype: int64
Final tuned submission shape: (451, 2)


,RecordID,Overall Seed
0,2020-21-AbileneChristian,14.0
1,2020-21-AlabamaA&M,67.0
2,2020-21-AlabamaSt.,64.0
3,2020-21-Alcorn,NaN
4,2020-21-Arkansas,2.0
5,2020-21-ArmyWestPoint,36.0
6,2020-21-Baylor,1.0
7,2020-21-Belmont,11.0
8,2020-21-Bethune-Cookman,22.0
9,2020-21-BostonCollege,48.0


In [60]:
# Fast finalize: keep ranks 1..68, set ranks > 68 to 69
TOP_N = 68
CAP_VALUE = -1

if "submit_tmp" in globals() and {"Season", "SelProb", "SeedPred"}.issubset(set(submit_tmp.columns)):
    temp = submit_tmp[["RecordID", "Season", "SelProb", "SeedPred"]].copy()
else:
    if "df_test2" not in globals():
        df_test2 = add_record_features(df_test)

    X_submit_fast = df_test2[candidate_features]
    temp = pd.DataFrame({
        "RecordID": df_test2["RecordID"],
        "Season": df_test2["RecordID"].str.extract(r"^(\d{4}-\d{2})")[0],
        "SelProb": clf.predict_proba(X_submit_fast)[:, 1],
        "SeedPred": reg.predict(X_submit_fast),
    })

blend_w = float(best_w) if "best_w" in globals() else 0.55
parts = []
for season, part in temp.groupby("Season", sort=True):
    part = part.copy()

    p = part["SelProb"]
    s = part["SeedPred"]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    score = blend_w * p_norm + (1.0 - blend_w) * (1.0 - s_norm)

    full_rank = score.rank(method="first", ascending=False).astype(int)
    part["Overall Seed"] = np.where(full_rank <= TOP_N, full_rank, CAP_VALUE)
    part["Overall Seed"] = part["Overall Seed"].astype(int)

    parts.append(part[["RecordID", "Season", "Overall Seed"]])

final_tmp = pd.concat(parts, ignore_index=True)
submission_tuned = final_tmp[["RecordID", "Overall Seed"]].sort_values("RecordID").reset_index(drop=True)

# Checks
assert submission_tuned.shape[0] == 451, f"Expected 451 rows, got {submission_tuned.shape[0]}"
assert submission_tuned["RecordID"].nunique() == 451, "RecordID duplicates found"
assert submission_tuned["Overall Seed"].notna().all(), "Found null seed values"

check_rank = final_tmp.copy()
positive = check_rank[check_rank["Overall Seed"] <= TOP_N].copy()
dupes = positive.duplicated(["Season", "Overall Seed"]).sum()
print("Duplicate ranks in 1..68 within season:", int(dupes))
print("Rows:", submission_tuned.shape[0], "| Count of 69:", int((submission_tuned["Overall Seed"] == CAP_VALUE).sum()))
print("Ranks 1..68 per season:")
print(check_rank.groupby("Season")["Overall Seed"].apply(lambda x: int((x <= TOP_N).sum())))

submission_tuned.to_csv("NCAA_Seed_Submission_tuned_cap69.csv", index=False)
submission_tuned.head(20)

Duplicate ranks in 1..68 within season: 106
Rows: 451 | Count of 69: 111
Ranks 1..68 per season:
Season
2020-21    89
2021-22    90
2022-23    91
2023-24    90
2024-25    91
Name: Overall Seed, dtype: int64


,RecordID,Overall Seed
0,2020-21-AbileneChristian,14
1,2020-21-AlabamaA&M,67
2,2020-21-AlabamaSt.,64
3,2020-21-Alcorn,-1
4,2020-21-Arkansas,2
5,2020-21-ArmyWestPoint,36
6,2020-21-Baylor,1
7,2020-21-Belmont,11
8,2020-21-Bethune-Cookman,22
9,2020-21-BostonCollege,48


In [61]:
# Advanced improvement: ensemble models + season-wise blend tuning + top68/0 submission
from sklearn.ensemble import ExtraTreesClassifier
from scipy.stats import spearmanr

TOP_N = 68

# Use engineered train/test already in notebook, else build from base frames.
if "train_df" not in globals():
    train_df = add_record_features(df.copy())
    train_df["MadeTournament"] = (~train_df["Bid Type"].isna()) & (train_df["Bid Type"].str.strip() != "")
    train_df["MadeTournament"] = train_df["MadeTournament"].astype(int)
    train_df["SeedNum"] = pd.to_numeric(train_df["Overall Seed"], errors="coerce")

if "df_test2" not in globals():
    df_test2 = add_record_features(df_test)

X_all_adv = train_df[candidate_features]
y_sel_adv = train_df["MadeTournament"]
seed_mask_adv = train_df["SeedNum"].notna()
X_seed_adv = train_df.loc[seed_mask_adv, candidate_features]
y_seed_adv = train_df.loc[seed_mask_adv, "SeedNum"]
X_submit_adv = df_test2[candidate_features]


def build_models():
    clf_rf = Pipeline([
        ("pre", clone(pre)),
        ("model", RandomForestClassifier(
            n_estimators=900,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    clf_et = Pipeline([
        ("pre", clone(pre)),
        ("model", ExtraTreesClassifier(
            n_estimators=900,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    reg_rf = Pipeline([
        ("pre", clone(pre)),
        ("model", RandomForestRegressor(
            n_estimators=1000,
            max_depth=8,
            min_samples_split=4,
            min_samples_leaf=1,
            max_features=0.5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    reg_et = Pipeline([
        ("pre", clone(pre)),
        ("model", ExtraTreesRegressor(
            n_estimators=1200,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    return clf_rf, clf_et, reg_rf, reg_et


def rank_with_blend(df_part, w_rank):
    p = df_part["SelProb"]
    s = df_part["SeedPred"]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    score = w_rank * p_norm + (1.0 - w_rank) * (1.0 - s_norm)
    return score.rank(method="first", ascending=False).astype(int)


# Tune ensemble and rank blend by season-wise holdout Spearman
seasons = sorted(train_df["Season"].dropna().unique())
sel_alphas = [0.4, 0.5, 0.6, 0.7]
seed_betas = [0.4, 0.5, 0.6, 0.7]
rank_ws = [0.5, 0.55, 0.6, 0.65]

best_cfg = None
best_score = -1.0

for a in sel_alphas:
    for b in seed_betas:
        for w in rank_ws:
            fold_scores = []

            for hold in seasons:
                tr_idx = np.where(train_df["Season"].values != hold)[0]
                va_idx = np.where(train_df["Season"].values == hold)[0]
                if len(tr_idx) == 0 or len(va_idx) == 0:
                    continue

                clf_rf, clf_et, reg_rf, reg_et = build_models()

                clf_rf.fit(X_all_adv.iloc[tr_idx], y_sel_adv.iloc[tr_idx])
                clf_et.fit(X_all_adv.iloc[tr_idx], y_sel_adv.iloc[tr_idx])

                tr_seed_idx = np.intersect1d(tr_idx, np.where(seed_mask_adv.values)[0])
                reg_rf.fit(X_all_adv.iloc[tr_seed_idx], train_df.iloc[tr_seed_idx]["SeedNum"])
                reg_et.fit(X_all_adv.iloc[tr_seed_idx], train_df.iloc[tr_seed_idx]["SeedNum"])

                va = train_df.iloc[va_idx][["RecordID", "Season", "SeedNum"]].copy()
                p_rf = clf_rf.predict_proba(X_all_adv.iloc[va_idx])[:, 1]
                p_et = clf_et.predict_proba(X_all_adv.iloc[va_idx])[:, 1]
                s_rf = reg_rf.predict(X_all_adv.iloc[va_idx])
                s_et = reg_et.predict(X_all_adv.iloc[va_idx])

                va["SelProb"] = a * p_rf + (1.0 - a) * p_et
                va["SeedPred"] = b * s_rf + (1.0 - b) * s_et
                va["PredRank"] = rank_with_blend(va, w)

                ev = va[va["SeedNum"].notna()].copy()
                if len(ev) > 1:
                    corr = spearmanr(ev["PredRank"], ev["SeedNum"], nan_policy="omit").correlation
                    if not np.isnan(corr):
                        fold_scores.append(float(corr))

            if fold_scores:
                score = float(np.mean(fold_scores))
                if score > best_score:
                    best_score = score
                    best_cfg = (a, b, w)

print("Best season-CV Spearman:", round(best_score, 4))
print("Best config (sel_alpha, seed_beta, rank_w):", best_cfg)

# Fit final ensemble models on all training data
sel_alpha, seed_beta, rank_w = best_cfg
clf_rf_f, clf_et_f, reg_rf_f, reg_et_f = build_models()
clf_rf_f.fit(X_all_adv, y_sel_adv)
clf_et_f.fit(X_all_adv, y_sel_adv)
reg_rf_f.fit(X_seed_adv, y_seed_adv)
reg_et_f.fit(X_seed_adv, y_seed_adv)

# Predict test set
p_rf_test = clf_rf_f.predict_proba(X_submit_adv)[:, 1]
p_et_test = clf_et_f.predict_proba(X_submit_adv)[:, 1]
s_rf_test = reg_rf_f.predict(X_submit_adv)
s_et_test = reg_et_f.predict(X_submit_adv)

submit_adv = pd.DataFrame({
    "RecordID": df_test2["RecordID"],
    "Season": df_test2["RecordID"].str.extract(r"^(\d{4}-\d{2})")[0],
})
submit_adv["SelProb"] = sel_alpha * p_rf_test + (1.0 - sel_alpha) * p_et_test
submit_adv["SeedPred"] = seed_beta * s_rf_test + (1.0 - seed_beta) * s_et_test

# Top 68 keep rank, rank > 68 becomes 0
out_parts = []
for season, part in submit_adv.groupby("Season", sort=True):
    part = part.copy()
    full_rank = rank_with_blend(part, rank_w)
    part["Overall Seed"] = np.where(full_rank <= TOP_N, full_rank, 0).astype(int)
    out_parts.append(part[["RecordID", "Season", "Overall Seed"]])

submission_advanced = pd.concat(out_parts, ignore_index=True)
submission_advanced = submission_advanced[["RecordID", "Overall Seed"]].sort_values("RecordID").reset_index(drop=True)

# Checks
assert submission_advanced.shape[0] == 451
assert submission_advanced["RecordID"].nunique() == 451
assert submission_advanced["Overall Seed"].notna().all()

tmp_check = submission_advanced.copy()
tmp_check["Season"] = tmp_check["RecordID"].str.extract(r"^(\d{4}-\d{2})")
pos = tmp_check[tmp_check["Overall Seed"] > 0]
print("Duplicate positive ranks within season:", int(pos.duplicated(["Season", "Overall Seed"]).sum()))
print("Positive ranks per season:")
print(tmp_check.groupby("Season")["Overall Seed"].apply(lambda x: int((x > 0).sum())))
print("Rows:", submission_advanced.shape[0], "| Zeros:", int((submission_advanced["Overall Seed"] == 0).sum()))

submission_advanced.to_csv("NCAA_Seed_Submission_advanced_top68_zero.csv", index=False)
submission_advanced.head(20)

Best season-CV Spearman: 0.9569
Best config (sel_alpha, seed_beta, rank_w): (0.4, 0.4, 0.5)
Duplicate positive ranks within season: 0
Positive ranks per season:
Season
2020-21    68
2021-22    68
2022-23    68
2023-24    68
2024-25    68
Name: Overall Seed, dtype: int64
Rows: 451 | Zeros: 111


,RecordID,Overall Seed
0,2020-21-AbileneChristian,14
1,2020-21-AlabamaA&M,67
2,2020-21-AlabamaSt.,0
3,2020-21-Alcorn,0
4,2020-21-Arkansas,2
5,2020-21-ArmyWestPoint,37
6,2020-21-Baylor,1
7,2020-21-Belmont,11
8,2020-21-Bethune-Cookman,22
9,2020-21-BostonCollege,45


In [ ]:
# Robust v2: feature ablation + new engineered features + season-wise CV model selection
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from scipy.stats import spearmanr

# Ensure engineered frames exist
if "train_df" not in globals():
    train_df = add_record_features(df.copy())
if "df_test2" not in globals():
    df_test2 = add_record_features(df_test)

# Ensure core targets exist
train_df = train_df.copy()
if "MadeTournament" not in train_df.columns:
    train_df["MadeTournament"] = (~train_df["Bid Type"].isna()) & (train_df["Bid Type"].str.strip() != "")
    train_df["MadeTournament"] = train_df["MadeTournament"].astype(int)
if "SeedNum" not in train_df.columns:
    train_df["SeedNum"] = pd.to_numeric(train_df["Overall Seed"], errors="coerce")
if "Season" not in train_df.columns:
    train_df["Season"] = train_df["RecordID"].str.extract(r"^(\d{4}-\d{2})")[0]

def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def _safe_div(a, b):
    return a / (b + 1e-9)

# Add a compact set of engineered features
for frame in (train_df, df_test2):
    frame["NET_Rank_num"] = _to_num(frame["NET Rank"])
    frame["PrevNET_num"] = _to_num(frame["PrevNET"])
    frame["AvgOppNET_Rank_num"] = _to_num(frame["AvgOppNET Rank"])
    frame["AvgOppNET_num"] = _to_num(frame["AvgOppNET"])
    frame["NETSOS_num"] = _to_num(frame["NETSOS"])
    frame["NETNonConfSOS_num"] = _to_num(frame["NETNonConfSOS"])

    frame["net_delta"] = frame["PrevNET_num"] - frame["NET_Rank_num"]
    frame["opp_gap"] = frame["AvgOppNET_Rank_num"] - frame["NET_Rank_num"]
    frame["sos_gap"] = frame["NETSOS_num"] - frame["NETNonConfSOS_num"]

    q1w = _to_num(frame.get("Quadrant1_wins", np.nan))
    q1l = _to_num(frame.get("Quadrant1_losses", np.nan))
    q2w = _to_num(frame.get("Quadrant2_wins", np.nan))
    q2l = _to_num(frame.get("Quadrant2_losses", np.nan))
    q3w = _to_num(frame.get("Quadrant3_wins", np.nan))
    q3l = _to_num(frame.get("Quadrant3_losses", np.nan))
    q4w = _to_num(frame.get("Quadrant4_wins", np.nan))
    q4l = _to_num(frame.get("Quadrant4_losses", np.nan))

    frame["q12_win_pct"] = _safe_div(q1w + q2w, q1w + q1l + q2w + q2l)
    frame["q34_loss_pct"] = _safe_div(q3l + q4l, q3w + q3l + q4w + q4l)
    frame["q1_to_q4_win_ratio"] = _safe_div(q1w + 1.0, q4w + 1.0)

# Feature scenarios
base_features = [c for c in candidate_features if c in train_df.columns]

reduced_drop = {
    "AvgOppNET", "AvgOppNET Rank",
    "overall_losses", "conf_losses", "nc_losses", "road_losses",
    "Quadrant2_losses", "Quadrant3_losses", "Quadrant4_losses"
}
reduced_features = [c for c in base_features if c not in reduced_drop]

engineered_new = [
    "net_delta", "opp_gap", "sos_gap",
    "q12_win_pct", "q34_loss_pct", "q1_to_q4_win_ratio"
]
expanded_features = base_features + [c for c in engineered_new if c in train_df.columns]

feature_scenarios = {
    "base": base_features,
    "reduced": reduced_features,
    "expanded": expanded_features,
}

y_sel_all = train_df["MadeTournament"].astype(int)
seed_mask = train_df["SeedNum"].notna()
y_seed_all = train_df.loc[seed_mask, "SeedNum"].astype(float)
seasons = sorted(train_df["Season"].dropna().unique())
blend_grid = [0.45, 0.55, 0.65, 0.75]

def make_preprocessor_for(features_local):
    num_cols_local = [c for c in features_local if c != "Conference"]
    cat_cols_local = [c for c in features_local if c == "Conference"]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols_local),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), cat_cols_local),
        ],
        remainder="drop",
    )

def rank_from_scores(sel_prob, seed_pred, weight):
    p = pd.Series(sel_prob)
    s = pd.Series(seed_pred)
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    score = weight * p_norm + (1.0 - weight) * (1.0 - s_norm)
    return score.rank(method="first", ascending=False).astype(int)

scenario_results = []
best_scenario = None
best_weight = None
best_score = -1.0

for scenario_name, feats in feature_scenarios.items():
    fold_corrs_by_w = {w: [] for w in blend_grid}
    X_all_local = train_df[feats]

    for hold in seasons:
        tr_idx = np.where(train_df["Season"].values != hold)[0]
        va_idx = np.where(train_df["Season"].values == hold)[0]
        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        pre_local = make_preprocessor_for(feats)
        clf_local = Pipeline([
            ("pre", clone(pre_local)),
            ("model", ExtraTreesClassifier(
                n_estimators=600,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )),
        ])
        reg_local = Pipeline([
            ("pre", clone(pre_local)),
            ("model", ExtraTreesRegressor(
                n_estimators=700,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            )),
        ])

        clf_local.fit(X_all_local.iloc[tr_idx], y_sel_all.iloc[tr_idx])
        tr_seed_idx = np.intersect1d(tr_idx, np.where(seed_mask.values)[0])
        reg_local.fit(X_all_local.iloc[tr_seed_idx], train_df.iloc[tr_seed_idx]["SeedNum"])

        val = train_df.iloc[va_idx][["SeedNum"]].copy()
        val["SelProb"] = clf_local.predict_proba(X_all_local.iloc[va_idx])[:, 1]
        val["SeedPred"] = reg_local.predict(X_all_local.iloc[va_idx])

        eval_val = val[val["SeedNum"].notna()].copy()
        if len(eval_val) < 2:
            continue

        for w in blend_grid:
            r = rank_from_scores(val["SelProb"], val["SeedPred"], w)
            eval_val["PredRank"] = r.loc[eval_val.index]
            corr = spearmanr(eval_val["PredRank"], eval_val["SeedNum"], nan_policy="omit").correlation
            if not np.isnan(corr):
                fold_corrs_by_w[w].append(float(corr))

    avg_by_w = {w: (float(np.mean(v)) if len(v) else -1.0) for w, v in fold_corrs_by_w.items()}
    local_best_w = max(avg_by_w, key=avg_by_w.get)
    local_best_score = avg_by_w[local_best_w]
    scenario_results.append((scenario_name, local_best_w, local_best_score, avg_by_w))

    if local_best_score > best_score:
        best_score = local_best_score
        best_scenario = scenario_name
        best_weight = local_best_w

print("Feature scenario CV results (Spearman, season-wise):")
for name, bw, bs, detail in scenario_results:
    clean_detail = {k: round(v, 4) for k, v in detail.items()}
    print(f"- {name}: best_w={bw}, best={round(bs, 4)}, by_w={clean_detail}")

print("\nSelected scenario:", best_scenario, "| best_w:", best_weight, "| CV Spearman:", round(best_score, 4))

# Fit final models on all data using best scenario
final_features = feature_scenarios[best_scenario]
X_train_final = train_df[final_features]
X_test_final = df_test2[final_features]
pre_final = make_preprocessor_for(final_features)

clf_final_v2 = Pipeline([
    ("pre", clone(pre_final)),
    ("model", ExtraTreesClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])
reg_final_v2 = Pipeline([
    ("pre", clone(pre_final)),
    ("model", ExtraTreesRegressor(
        n_estimators=900,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
])

clf_final_v2.fit(X_train_final, y_sel_all)
reg_final_v2.fit(X_train_final.loc[seed_mask], y_seed_all)

# Build per-season ranked submissions
pred_sel = clf_final_v2.predict_proba(X_test_final)[:, 1]
pred_seed = reg_final_v2.predict(X_test_final)
submit_v2 = pd.DataFrame({
    "RecordID": df_test2["RecordID"],
    "Season": df_test2["RecordID"].str.extract(r"^(\d{4}-\d{2})")[0],
    "SelProb": pred_sel,
    "SeedPred": pred_seed,
})

TOP_N = 68
parts_zero = []
parts_69 = []
for season, part in submit_v2.groupby("Season", sort=True):
    part = part.copy()
    full_rank = rank_from_scores(part["SelProb"], part["SeedPred"], best_weight)
    part["Overall Seed 0"] = np.where(full_rank <= TOP_N, full_rank, 0).astype(int)
    part["Overall Seed 69"] = np.where(full_rank <= TOP_N, full_rank, 69).astype(int)
    parts_zero.append(part[["RecordID", "Season", "Overall Seed 0"]])
    parts_69.append(part[["RecordID", "Season", "Overall Seed 69"]])

sub_zero = pd.concat(parts_zero, ignore_index=True)
sub_69 = pd.concat(parts_69, ignore_index=True)

submission_feature_v2_zero = sub_zero[["RecordID", "Overall Seed 0"]].rename(columns={"Overall Seed 0": "Overall Seed"}).sort_values("RecordID").reset_index(drop=True)
submission_feature_v2_69 = sub_69[["RecordID", "Overall Seed 69"]].rename(columns={"Overall Seed 69": "Overall Seed"}).sort_values("RecordID").reset_index(drop=True)

# Checks
for sname, sframe in [("zero", submission_feature_v2_zero), ("69", submission_feature_v2_69)]:
    assert sframe.shape[0] == 451, f"{sname}: expected 451 rows"
    assert sframe["RecordID"].nunique() == 451, f"{sname}: duplicate RecordID"
    assert sframe["Overall Seed"].notna().all(), f"{sname}: null seed values"

print("\nRows (zero):", submission_feature_v2_zero.shape[0], "| zeros:", int((submission_feature_v2_zero["Overall Seed"] == 0).sum()))
print("Rows (69):", submission_feature_v2_69.shape[0], "| 69s:", int((submission_feature_v2_69["Overall Seed"] == 69).sum()))

submission_feature_v2_zero.to_csv("NCAA_Seed_Submission_feature_v2_top68_zero.csv", index=False)
submission_feature_v2_69.to_csv("NCAA_Seed_Submission_feature_v2_top68_69.csv", index=False)

submission_feature_v2_zero.head(20)

c:\Users\harsh\anaconda3\envs\HW1env\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['opp_gap']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\harsh\anaconda3\envs\HW1env\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['opp_gap']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\harsh\anaconda3\envs\HW1env\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['opp_gap']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\harsh\anaconda3\envs\HW1env\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['opp_gap']. At least one non-missing value is needed for imputation with strategy='median'.
 

Feature scenario CV results (Spearman, season-wise):
- base: best_w=0.45, best=0.9619, by_w={0.45: 0.9619, 0.55: 0.9542, 0.65: 0.9472, 0.75: 0.9384}
- reduced: best_w=0.45, best=0.9595, by_w={0.45: 0.9595, 0.55: 0.9547, 0.65: 0.9494, 0.75: 0.9396}
- expanded: best_w=0.45, best=0.9613, by_w={0.45: 0.9613, 0.55: 0.9553, 0.65: 0.9486, 0.75: 0.9402}

Selected scenario: base | best_w: 0.45 | CV Spearman: 0.9619

Rows (zero): 451 | zeros: 111
Rows (69): 451 | 69s: 111


,RecordID,Overall Seed
0,2020-21-AbileneChristian,14
1,2020-21-AlabamaA&M,0
2,2020-21-AlabamaSt.,0
3,2020-21-Alcorn,0
4,2020-21-Arkansas,2
5,2020-21-ArmyWestPoint,37
6,2020-21-Baylor,1
7,2020-21-Belmont,11
8,2020-21-Bethune-Cookman,23
9,2020-21-BostonCollege,45
